In [7]:
import pandas as pd
df = pd.read_csv('data/raw_claims.csv')
print(df.head())
print(df.head)
print(df.columns.tolist())
print(df.isnull().sum())
print(df[df['Claim_Status'].isnull()]['Claim_Count'].value_counts())

  Policy_ID Policy_Number Customer_ID State     Region              Product  \
0  POL00001     PN-205907   CUST00001    MD      South                 Auto   
1  POL00002     PN-944151   CUST00002    MA  Northeast                 Auto   
2  POL00003     PN-711878   CUST00003    CT  Northeast  Commercial Property   
3  POL00004     PN-654816   CUST00004    OH    Midwest  Commercial Property   
4  POL00005     PN-624902   CUST00005    OH    Midwest  Commercial Property   

      Line_of_Business            Agent     Underwriter Policy_Status  ...  \
0        Personal Auto  Kimberly Garcia   Paul Williams        Active  ...   
1        Personal Auto   Kimberly Allen     Mark Miller        Active  ...   
2  Commercial Property     Ashley Brown     Donna Davis        Active  ...   
3  Commercial Property     Thomas Brown       Brian Lee        Active  ...   
4  Commercial Property     Angela Allen  Donna Anderson        Active  ...   

  Claim_Count Claim_Amount Paid_Claim_Amount  Outstandin

In [8]:
df['Claim_Status'] = df['Claim_Status'].fillna('No Claim')
df['Claim_Type'] = df['Claim_Type'].fillna('No Claim')

In [9]:
print(df['Claim_Status'].value_counts())

Claim_Status
No Claim    314
Closed      123
Open         44
Reopened     19
Name: count, dtype: int64


In [10]:
df['Effective_Date'] = pd.to_datetime(df['Effective_Date'])
df['Expiration_Date'] = pd.to_datetime(df['Expiration_Date'])
df['Loss_Date'] = pd.to_datetime(df['Loss_Date'])
df['Reported_Date'] = pd.to_datetime(df['Reported_Date'])

In [11]:
bad_dates = df[df['Loss_Date'] < df['Effective_Date']]
print(f"Number of claims with Loss_Date before Effective_Date: {len(bad_dates)}")

Number of claims with Loss_Date before Effective_Date: 0


In [12]:
df = pd.read_csv('data/raw_claims.csv')
df['Claim_Status'] = df['Claim_Status'].fillna('No Claim')
df['Claim_Type'] = df['Claim_Type'].fillna('No Claim')

In [13]:
df['Effective_Date'] = pd.to_datetime(df['Effective_Date'])
df['Expiration_Date'] = pd.to_datetime(df['Expiration_Date'])
df['Loss_Date'] = pd.to_datetime(df['Loss_Date'])
df['Reported_Date'] = pd.to_datetime(df['Reported_Date'])

In [14]:
bad_dates = df[df['Loss_Date'] < df['Effective_Date']]
print(f"Number of claims with Loss_Date before Effective_Date: {len(bad_dates)}")

Number of claims with Loss_Date before Effective_Date: 0


In [15]:
df = pd.read_csv('data/raw_claims.csv')
df['Claim_Status'] = df['Claim_Status'].fillna('No Claim')
df['Claim_Type'] = df['Claim_Type'].fillna('No Claim')
df['Effective_Date'] = pd.to_datetime(df['Effective_Date'])
df['Expiration_Date'] = pd.to_datetime(df['Expiration_Date'])
df['Loss_Date'] = pd.to_datetime(df['Loss_Date'])
df['Reported_Date'] = pd.to_datetime(df['Reported_Date'])
bad_dates = df[df['Loss_Date'] < df['Effective_Date']]
print(f"Number of claims with Loss_Date before Effective_Date: {len(bad_dates)}")

Number of claims with Loss_Date before Effective_Date: 0


In [16]:
duplicate_count = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_count}")

Number of duplicate rows: 0


In [23]:
df['Loss_Ratio'] = df['Claim_Amount']/df['Earned_Premium']
loss_ratio_by_product = df.groupby('Product') .agg(
    Total_Claims=('Claim_Amount', 'sum'),
    Total_Premium=('Earned_Premium','sum')
)
loss_ratio_by_product['Loss_Ratio']=loss_ratio_by_product['Total_Claims']/loss_ratio_by_product['Total_Premium']
print(loss_ratio_by_product)

                     Total_Claims  Total_Premium  Loss_Ratio
Product                                                     
Auto                    825969.31      204343.40    4.042065
Commercial Auto         356708.95      398181.14    0.895846
Commercial Property     109600.28      365862.02    0.299567
Home                    498420.44      168433.94    2.959145


In [25]:
loss_ratio_by_region = df.groupby('Region') .agg(
    Total_Claims=('Claim_Amount', 'sum'),
    Total_Premium=('Earned_Premium','sum')
)
loss_ratio_by_region['Loss_Ratio']=loss_ratio_by_region['Total_Claims']/loss_ratio_by_region['Total_Premium']
print(loss_ratio_by_region)

           Total_Claims  Total_Premium  Loss_Ratio
Region                                            
Midwest       283182.80      158964.72    1.781419
Northeast     464956.79      405186.74    1.147512
South         855440.97      447860.49    1.910061
West          187118.42      124808.55    1.499244


In [27]:
loss_ratio_by_product_region = df.groupby(['Product','Region']) .agg(
    Total_Claims=('Claim_Amount', 'sum'),
    Total_Premium=('Earned_Premium','sum')
)
loss_ratio_by_product_region['Loss_Ratio']=loss_ratio_by_product_region['Total_Claims']/loss_ratio_by_product_region['Total_Premium']
print(loss_ratio_by_product_region)

                               Total_Claims  Total_Premium  Loss_Ratio
Product             Region                                            
Auto                Midwest       135341.49       29660.85    4.562967
                    Northeast     174691.91       60008.13    2.911137
                    South         416713.98      101929.23    4.088268
                    West           99221.93       12745.19    7.785049
Commercial Auto     Midwest        44969.54       48220.60    0.932579
                    Northeast      93804.85      150431.67    0.623571
                    South         164878.09      160804.87    1.025330
                    West           53056.47       38724.00    1.370119
Commercial Property Midwest        45271.82       67902.16    0.666721
                    Northeast      31397.30      128209.58    0.244890
                    South          20580.95      109788.20    0.187460
                    West           12350.21       59962.08    0.205967
Home  

In [ ]:
print(loss_ratio_by_product_region.sort_values('Loss_Ratio', ascending=False))

In [28]:
df.to_csv('data/cleaned_claims.csv',index=False)
print("Saved cleaned_claims.csv")

Saved cleaned_claims.csv


In [30]:
from sqlalchemy import create_engine
engine = create_engine('sqlite:///pnc_insurance.db')
df.to_sql('claims', engine, if_exists='replace', index=False)
print("Data loaded into SQLite database")

Data loaded into SQLite database


In [31]:
query = """
SELECT Product,
       SUM(Claim_Amount) AS Total_Claims,
       SUM(Earned_Premium) AS Total_Premium,
       ROUND(SUM(Claim_Amount) * 1.0 / SUM(Earned_Premium), 3) AS Loss_Ratio
FROM claims
GROUP BY Product
ORDER BY Loss_Ratio DESC
"""
result = pd.read_sql(query, engine)
print(result)

               Product  Total_Claims  Total_Premium  Loss_Ratio
0                 Auto     825969.31      204343.40       4.042
1                 Home     498420.44      168433.94       2.959
2      Commercial Auto     356708.95      398181.14       0.896
3  Commercial Property     109600.28      365862.02       0.300


In [32]:
query_region = """
SELECT Region,
       SUM(Claim_Amount) AS Total_Claims,
       SUM(Earned_Premium) AS Total_Premium,
       ROUND(SUM(Claim_Amount) * 1.0 / SUM(Earned_Premium), 3) AS Loss_Ratio
FROM claims
GROUP BY Region
ORDER BY Loss_Ratio DESC
"""
result_region = pd.read_sql(query_region, engine)
print(result_region)

      Region  Total_Claims  Total_Premium  Loss_Ratio
0      South     855440.97      447860.49       1.910
1    Midwest     283182.80      158964.72       1.781
2       West     187118.42      124808.55       1.499
3  Northeast     464956.79      405186.74       1.148


In [33]:
query_window = """
WITH product_loss AS (
    SELECT Product,
           SUM(Claim_Amount) AS Total_Claims,
           SUM(Earned_Premium) AS Total_Premium
    FROM claims
    GROUP BY Product
)
SELECT Product,
       Total_Claims,
       Total_Premium,
       ROUND(Total_Claims * 1.0 / Total_Premium, 3) AS Loss_Ratio,
       RANK() OVER (ORDER BY Total_Claims * 1.0 / Total_Premium DESC) AS Risk_Rank
FROM product_loss
"""
result_window = pd.read_sql(query_window, engine)
print(result_window)

               Product  Total_Claims  Total_Premium  Loss_Ratio  Risk_Rank
0                 Auto     825969.31      204343.40       4.042          1
1                 Home     498420.44      168433.94       2.959          2
2      Commercial Auto     356708.95      398181.14       0.896          3
3  Commercial Property     109600.28      365862.02       0.300          4


In [34]:
query_window_region = """
WITH region_loss AS (
    SELECT Region,
           SUM(Claim_Amount) AS Total_Claims,
           SUM(Earned_Premium) AS Total_Premium
    FROM claims
    GROUP BY Region
)
SELECT Region,
       Total_Claims,
       Total_Premium,
       ROUND(Total_Claims * 1.0 / Total_Premium, 3) AS Loss_Ratio,
       RANK() OVER (ORDER BY Total_Claims * 1.0 / Total_Premium DESC) AS Risk_Rank
FROM region_loss
"""
result_window_region = pd.read_sql(query_window_region, engine)
print(result_window_region)

      Region  Total_Claims  Total_Premium  Loss_Ratio  Risk_Rank
0      South     855440.97      447860.49       1.910          1
1    Midwest     283182.80      158964.72       1.781          2
2       West     187118.42      124808.55       1.499          3
3  Northeast     464956.79      405186.74       1.148          4
